In [ ]:
from glob import glob
import json
import os
import base64
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

# 프로젝트 루트의 .env 자동 탐색 (실행 위치와 무관하게 동작)
load_dotenv(find_dotenv())
api_key = os.getenv("OPENAI_API_KEY")

# OpenRouter 사용 (키도 OpenRouter 키)
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

# 경로 설정 (.env 위치 = 프로젝트 루트 기준)
ROOT = os.path.dirname(find_dotenv())
IMAGE_DIR = os.path.join(ROOT, "chapter03", "data", "images")
OUT_DIR = os.path.join(ROOT, "chapter03", "data")


def encoding_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def image_quiz(image_path, n_trial=0, max_trial=3):
    if n_trial >= max_trial:
        raise Exception("Maximum number of trials exceeded.")

    base_image = encoding_image(image_path)
    ext = os.path.splitext(image_path)[1].lower().lstrip(".")
    mime = "jpeg" if ext in ("jpg", "jpeg") else ext

    quiz_prompt = """
    제공된 이미지를 바탕으로, 다음과 같은 양식으로 퀴즈를 만들어주세요.
    정답은 1~4 중 하나만 해당하도록 출제하세요.
    토익 리스팅 문제 스타일로 문제를 만ㄷ르어 주세요.
    아래는 예시입니다.
    ----- 예시 -----

    Q : 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
    - (1) 베이커리에서 사람들이 빠을 사고 있는 모습이 담겨 있습니다.
    - (2) 맨 앞에 서있는 사람은 빨간색 셔츠를 입고 있습니다.
    - (3) 기차를 타기 위해 줄을 서 있는 사람들이 있습니다.
    - (4) 점원은 노란색 티셔츠를 입고 있습니다.

    Listening : Which of the following descriptions of the image is incorrect?
    - (1) It shows people buying bread at a bakery.
    - (2) The person standing at the front is wearing a red shirt.
    - (3) There are people lining up to take a train.
    - (4) The clerk is wearing a yellow T-shirt.

    정답: (4) 점원은 노란색 티셔츠가 아닌 파란색 티셔츠를 입고 있습니다.
    (주의: 정답은 1~4 중 하나만 선택되도록 출제하세요.)
    """

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": quiz_prompt,
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/{mime};base64,{base_image}"
                    },
                },
            ],
        }
    ]

    try:
        response = client.chat.completions.create(
            model="openai/gpt-4o",
            messages=messages,
        )
    except Exception as e:
        print("failed\n" + str(e))
        return image_quiz(image_path, n_trial + 1)

    content = response.choices[0].message.content

    if content and "Listening" in content:
        return content, True
    else:
        return image_quiz(image_path, n_trial + 1)


# 처리할 이미지 목록
sample_images = sorted(
    g for g in glob(os.path.join(IMAGE_DIR, "*"))
    if g.lower().endswith((".png", ".jpg", ".jpeg"))
)


def safe_quiz(image_path):
    """병렬 실행용 래퍼: 실패해도 전체가 멈추지 않도록 예외를 흡수."""
    try:
        return image_quiz(image_path)
    except Exception as e:
        print(f"{os.path.basename(image_path)} 실패: {e}")
        return None, False


# 모든 이미지를 병렬로 동시에 호출 (12장이 거의 1장 시간으로 단축)
with ThreadPoolExecutor(max_workers=8) as executor:
    results = list(executor.map(safe_quiz, sample_images))

# 결과를 이미지 순서대로 정리 (번호 매기기)
txt = ""
eng_dict = []
no = 1

for image_path, (q, is_succeed) in zip(sample_images, results):
    if not is_succeed:
        continue

    filename = os.path.basename(image_path)
    txt += f"## 문제 {no}\n\n"
    txt += f"![image](images/{filename})\n\n"
    txt += q + "\n\n----------------\n\n"

    # 영어 문제만 추출 ("Listening :" / "Listening:" 모두 대응)
    eng = q.split("Listening")[1].lstrip(" :").split("정답")[0].strip()
    eng_dict.append({"no": no, "eng": eng, "img": filename})

    no += 1

# 마크다운 파일로 저장
with open(os.path.join(OUT_DIR, "image_quiz_eng.md"), "w", encoding="utf-8") as f:
    f.write(txt)

# json 파일로 저장
with open(os.path.join(OUT_DIR, "image_quiz_eng.json"), "w", encoding="utf-8") as f:
    json.dump(eng_dict, f, ensure_ascii=False, indent=4)

print(f"완료: {len(eng_dict)}개 문제 생성 -> {OUT_DIR}")
